## Segmentation Model --- using U-Net

Here, the training image is the input and the mask image data is the output. We train our model using this data and then use the test data to predict the mask images

In [1]:
#Loading dependencies
import tensorflow as tf
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img,img_to_array

D:\IISc\tf_env\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Setting  the  image size and it should match the mask size
img_size = (128, 128) 

# Creating the folders
image_dir = 'D:\\IISc\\projects\\MLDS - Leaf Disease Detection\\Dataset\\train\\images'
mask_dir = 'D:\\IISc\\projects\\MLDS - Leaf Disease Detection\\Dataset\\train\\masks'

image_files = sorted(os.listdir(image_dir))
mask_files = sorted(os.listdir(mask_dir))

# Creating empty lists to store training data
X_train = []
Y_train = []

# Loop over all image-mask pairs
for img_file, mask_file in zip(image_files, mask_files):
    # Load image
    img = load_img(os.path.join(image_dir, img_file), target_size=img_size)
    img = img_to_array(img) / 255.0  # Normalize to 0–1
    X_train.append(img)

    # Load mask
    mask = load_img(os.path.join(mask_dir, mask_file), target_size=img_size, color_mode='grayscale')
    mask = img_to_array(mask) / 255.0
    mask = (mask > 0.5).astype(np.float32)  # Convert to binary (0 or 1)
    Y_train.append(mask)

# Convert to numpy arrays
X_train = np.array(X_train)
Y_train = np.array(Y_train)

# Check shapes
print("Image shape:", X_train.shape)
print("Mask shape:", Y_train.shape)


print("Mask unique values:", np.unique(Y_train))


Image shape: (909, 128, 128, 3)
Mask shape: (909, 128, 128, 1)
Mask unique values: [0. 1.]


## Building the U-Net model

In [3]:
from tensorflow.keras import layers, models


#Building the Segmentation model with separate layers for Batch Normalization ,padding and pooling

def build_unet(input_shape=(128, 128, 3)):
    inputs = layers.Input(input_shape)

    # Encoder (Downsampling)
    c1 = layers.Conv2D(16, 3, padding='same')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('relu')(c1)
    c1 = layers.Conv2D(16, 3, padding='same')(c1)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('relu')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(32, 3, padding='same')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('relu')(c2)
    c2 = layers.Conv2D(32, 3, padding='same')(c2)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('relu')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(64, 3, padding='same')(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('relu')(c3)
    c3 = layers.Conv2D(64, 3, padding='same')(c3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('relu')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(128, 3, padding='same')(p3)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('relu')(c4)
    c4 = layers.Conv2D(128, 3, padding='same')(c4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('relu')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Bottleneck
    c5 = layers.Conv2D(256, 3, padding='same')(p4)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Activation('relu')(c5)
    c5 = layers.Conv2D(256, 3, padding='same')(c5)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Activation('relu')(c5)

    c5 = layers.Dropout(0.3)(c5)

    # Decoder (Upsampling)
    u6 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(128, 3, padding='same')(u6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Activation('relu')(c6)
    c6 = layers.Conv2D(128, 3, padding='same')(c6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Activation('relu')(c6)

    c6 = layers.Dropout(0.2)(c6)

    u7 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(64, 3, padding='same')(u7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Activation('relu')(c7)
    c7 = layers.Conv2D(64, 3, padding='same')(c7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Activation('relu')(c7)

    c7 = layers.Dropout(0.2)(c7)

    u8 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(32, 3, padding='same')(u8)
    c8 = layers.BatchNormalization()(c8)
    c8 = layers.Activation('relu')(c8)
    c8 = layers.Conv2D(32, 3, padding='same')(c8)
    c8 = layers.BatchNormalization()(c8)
    c8 = layers.Activation('relu')(c8)

    c8 = layers.Dropout(0.2)(c8)

    u9 = layers.Conv2DTranspose(16, 2, strides=2, padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(16, 3, padding='same')(u9)
    c9 = layers.BatchNormalization()(c9)
    c9 = layers.Activation('relu')(c9)
    c9 = layers.Conv2D(16, 3, padding='same')(c9)
    c9 = layers.BatchNormalization()(c9)
    c9 = layers.Activation('relu')(c9)

    c9 = layers.Dropout(0.2)(c9)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c9)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model


Training the model

In [4]:
from tensorflow.keras import backend as K
import gc


K.clear_session()
gc.collect()


model = build_unet()
#model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


#Defining a new loss function names dice loss inorder to imporve model performance    
def dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)   
    y_pred = tf.cast(y_pred, tf.float32)
    numerator = 2 * tf.reduce_sum(y_true * y_pred)
    denominator = tf.reduce_sum(y_true + y_pred)
    return 1 - numerator / (denominator + 1e-6)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

#model.compile(optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=0.0001),loss=bce_dice_loss,metrics=['accuracy'])#Using Adam Optimizer with a learning rate of 0.0001
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 128, 128, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 128, 128, 16)      │             448 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 128, 128, 16)      │              64 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 128, 128, 16)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 128, 128, 16)      │           2,320 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 128, 128, 16)      │              64 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 128, 128, 16)      │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 64, 64, 16)        │               0 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 64, 64, 32)        │           4,640 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 64, 64, 32)        │             128 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 64, 64, 32)        │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 64, 64, 32)        │           9,248 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 64, 64, 32)        │             128 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 64, 64, 32)        │               0 │ batch_normalization_3[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_1               │ (None, 32, 32, 32)        │               0 │ activation_3[0][0]         │
│ (MaxPooling2D)                │                           │               

 Total params: 1,946,993 (7.43 MB)

 Trainable params: 1,944,049 (7.42 MB)

 Non-trainable params: 2,944 (11.50 KB)

In [5]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True) #Hyperparameter tuning by implementing early stopping
checkpoint = ModelCheckpoint('best_unet_model.h5', save_best_only=True)

# Train the model
history = model.fit(
    X_train,
    Y_train,
    validation_split=0.2,    # 80% training, 20% validation
    batch_size=8,
    epochs=20,
    callbacks=[early_stop, checkpoint]
)


Epoch 1/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.4725 - loss: 1.6892

91/91 ━━━━━━━━━━━━━━━━━━━━ 35s 227ms/step - accuracy: 0.5296 - loss: 1.6274 - val_accuracy: 0.8155 - val_loss: 1.4954
Epoch 2/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.7221 - loss: 1.4461

91/91 ━━━━━━━━━━━━━━━━━━━━ 19s 212ms/step - accuracy: 0.7768 - loss: 1.3805 - val_accuracy: 0.8745 - val_loss: 1.4212
Epoch 3/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.8572 - loss: 1.2488

91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 197ms/step - accuracy: 0.8706 - loss: 1.2248 - val_accuracy: 0.8887 - val_loss: 1.3002
Epoch 4/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step - accuracy: 0.8931 - loss: 1.1866

91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 201ms/step - accuracy: 0.8906 - loss: 1.1811 - val_accuracy: 0.8881 - val_loss: 1.2042
Epoch 5/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step - accuracy: 0.9021 - loss: 1.1334

91/91 ━━━━━━━━━━━━━━━━━━━━ 19s 203ms/step - accuracy: 0.8999 - loss: 1.1309 - val_accuracy: 0.9128 - val_loss: 1.1042
Epoch 6/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step - accuracy: 0.9009 - loss: 1.1109

91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 198ms/step - accuracy: 0.9008 - loss: 1.0977 - val_accuracy: 0.9092 - val_loss: 1.0647
Epoch 7/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.9026 - loss: 1.0629

91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 194ms/step - accuracy: 0.9044 - loss: 1.0626 - val_accuracy: 0.9209 - val_loss: 0.9780
Epoch 8/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 192ms/step - accuracy: 0.9078 - loss: 1.0149 - val_accuracy: 0.9033 - val_loss: 1.0000
Epoch 9/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.9152 - loss: 0.9874

91/91 ━━━━━━━━━━━━━━━━━━━━ 19s 211ms/step - accuracy: 0.9074 - loss: 0.9926 - val_accuracy: 0.9252 - val_loss: 0.9232
Epoch 10/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 193ms/step - accuracy: 0.9108 - loss: 0.9591 - val_accuracy: 0.9176 - val_loss: 0.9259
Epoch 11/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy: 0.9138 - loss: 0.9628

91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 197ms/step - accuracy: 0.9161 - loss: 0.9372 - val_accuracy: 0.9261 - val_loss: 0.8688
Epoch 12/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 19s 207ms/step - accuracy: 0.9158 - loss: 0.9062 - val_accuracy: 0.9074 - val_loss: 0.8972
Epoch 13/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 18s 197ms/step - accuracy: 0.9204 - loss: 0.8832 - val_accuracy: 0.9273 - val_loss: 0.8752
Epoch 14/20
91/91 ━━━━━━━━━━━━━━━━━━━━ 17s 189ms/step - accuracy: 0.9262 - loss: 0.8473 - val_accuracy: 0.9201 - val_loss: 0.9558


In [6]:
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array   #Applying the model on the test data
import numpy as np

test_dir = 'D:\\IISc\\projects\\MLDS - Leaf Disease Detection\\Dataset\\test\\images'
test_files = sorted(os.listdir(test_dir))
img_size = (128, 128)

X_test = []

for filename in test_files:
    img_path = os.path.join(test_dir, filename)
    img = load_img(img_path, target_size=img_size)
    img = img_to_array(img) / 255.0
    X_test.append(img)

X_test = np.array(X_test)
print("Test shape:", X_test.shape)


Test shape: (390, 128, 128, 3)


In [7]:
def mask2rle(img):   #using the code given in the question to convert the masks to rle 
    # https://www.kaggle.com/code/paulorzp/rle-functions-run-lenght-encode-decode
    pixels = img.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

def rle2mask(mask_rle, shape=(256,256)):
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape).T


In [8]:
import cv2    
 

mask_preds = model.predict(X_test, batch_size=8)



rle_outputs = []

for mask in mask_preds:
    binary_mask = (mask.squeeze() > 0.5).astype(np.uint8)

    # Resize to 256×256 if your masks are 128×128
    resized_mask = cv2.resize(binary_mask, (256, 256), interpolation=cv2.INTER_NEAREST)

    if resized_mask.sum() == 0:
        rle_outputs.append("Healthy")  # No disease pixels
    else:
        rle_outputs.append(mask2rle(resized_mask))  # Convert to RLE


mask_preds[0]


49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step


array([[[0.34303984],
        [0.18897717],
        [0.18087907],
        ...,
        [0.21517777],
        [0.20051095],
        [0.24544758]],

       [[0.22679502],
        [0.10452248],
        [0.10663436],
        ...,
        [0.10684783],
        [0.12467681],
        [0.14867853]],

       [[0.18884428],
        [0.09478098],
        [0.12177234],
        ...,
        [0.13332832],
        [0.1396394 ],
        [0.18009049]],

       ...,

       [[0.18824896],
        [0.10159607],
        [0.11045519],
        ...,
        [0.12243509],
        [0.15078034],
        [0.13730378]],

       [[0.16411623],
        [0.12963827],
        [0.14524719],
        ...,
        [0.13564833],
        [0.17194885],
        [0.16054776]],

       [[0.2357551 ],
        [0.15818751],
        [0.22712697],
        ...,
        [0.1706847 ],
        [0.21940915],
        [0.23873118]]], shape=(128, 128, 1), dtype=float32)

In [10]:
import pandas as pd
import os

# Get test image file names
file_names = sorted(os.listdir('D:\\IISc\\projects\\MLDS - Leaf Disease Detection\\Dataset\\test\\images'))  # or your actual test folder

submission_df = pd.DataFrame({
    'id': file_names,
    'segmentation_pred': rle_outputs
})

submission_df.to_csv("segmentation_final.csv", index=False)  #Saving the csv file with segmentation output
